In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# --- 1. DATA GENERATION ---
def generate_data(n_points=500, mode='cnn'):
    """
    Generates gradient vectors.
    mode='cnn' -> Isotropic (Ball shape)
    mode='transformer' -> Anisotropic (Cone shape)
    """
    if mode == 'cnn':
        # Gaussian distribution centered at 0 (The "Ball")
        # Gradients point in all random directions
        data = np.random.randn(n_points, 3)
        
    elif mode == 'transformer':
        # Strong mean direction (The "Cone")
        # Gradients are clustered around the vector [1, 1, 1]
        mean_vec = np.array([1.0, 1.0, 1.0])
        noise = np.random.randn(n_points, 3) * 0.2  # Small noise
        data = mean_vec + noise
        
    # Normalize vectors to unit length for clearer visualization
    norms = np.linalg.norm(data, axis=1, keepdims=True)
    return data / norms

def get_random_cavs(n_cavs=1000):
    # Random vectors uniformly distributed on the unit sphere
    cavs = np.random.randn(n_cavs, 3)
    norms = np.linalg.norm(cavs, axis=1, keepdims=True)
    return cavs / norms

# --- 2. CALCULATE SCORES ---
def calculate_tcav_scores(gradients, cavs):
    scores = []
    # For each random CAV, check how many gradients align with it
    for cav in cavs:
        # Dot product: (N_grads, 3) @ (3,) -> (N_grads,)
        sensitivities = np.dot(gradients, cav)
        
        # TCAV Score = Fraction of positive sensitivities
        score = np.sum(sensitivities > 0) / len(sensitivities)
        scores.append(score)
    return scores

# --- 3. RUN SIMULATION ---
n_grads = 500
n_cavs = 2000

# Generate Gradients (The Data)
grads_cnn = generate_data(n_grads, mode='cnn')
grads_transformer = generate_data(n_grads, mode='transformer')

# Generate Random CAVs (The Probes)
random_cavs = get_random_cavs(n_cavs)

# Compute Scores
scores_cnn = calculate_tcav_scores(grads_cnn, random_cavs)
scores_transformer = calculate_tcav_scores(grads_transformer, random_cavs)

# --- 4. PLOTTING ---
fig = plt.figure(figsize=(18, 10))

# --- PLOT A: CNN (ISOTROPIC) ---
ax1 = fig.add_subplot(2, 2, 1, projection='3d')
# Plot Gradients (Blue Dots)
ax1.scatter(grads_cnn[:,0], grads_cnn[:,1], grads_cnn[:,2], 
            c='blue', alpha=0.6, s=10, label='Gradients (Data)')
# Plot a few Random CAVs (Red Arrows) to show they probe all directions
for i in range(50):
    v = random_cavs[i]
    ax1.quiver(0,0,0, v[0], v[1], v[2], color='red', length=1.2, arrow_length_ratio=0.1)
ax1.set_title("CNN Gradients: Isotropic 'Ball'\n(Gradients point everywhere)", fontsize=14, fontweight='bold')
ax1.set_xlim([-1,1]); ax1.set_ylim([-1,1]); ax1.set_zlim([-1,1])
ax1.legend()

# --- PLOT B: TRANSFORMER (ANISOTROPIC) ---
ax2 = fig.add_subplot(2, 2, 2, projection='3d')
# Plot Gradients (Purple Dots)
ax2.scatter(grads_transformer[:,0], grads_transformer[:,1], grads_transformer[:,2], 
            c='purple', alpha=0.6, s=10, label='Gradients (Data)')
# Plot a few Random CAVs (Red Arrows)
for i in range(50):
    v = random_cavs[i]
    ax2.quiver(0,0,0, v[0], v[1], v[2], color='red', length=1.2, arrow_length_ratio=0.1)
ax2.set_title("Transformer Gradients: Anisotropic 'Cone'\n(Gradients cluster in one direction)", fontsize=14, fontweight='bold')
ax2.set_xlim([-1,1]); ax2.set_ylim([-1,1]); ax2.set_zlim([-1,1])
ax2.legend()

# --- PLOT C: CNN SCORES ---
ax3 = fig.add_subplot(2, 2, 3)
sns.histplot(scores_cnn, bins=50, color='blue', kde=True, ax=ax3)
ax3.axvline(0.5, color='black', linestyle='--', label='Random Chance (0.5)')
ax3.set_title("CNN Score Distribution: Normal / Bell Curve", fontsize=14)
ax3.set_xlabel("TCAV Score")
ax3.set_ylabel("Count")
ax3.legend()

# --- PLOT D: TRANSFORMER SCORES ---
ax4 = fig.add_subplot(2, 2, 4)
sns.histplot(scores_transformer, bins=50, color='purple', kde=True, ax=ax4)
ax4.axvline(0.5, color='black', linestyle='--', label='Random Chance (0.5)')
ax4.set_title("Transformer Score Distribution: Bimodal / U-Shape", fontsize=14)
ax4.set_xlabel("TCAV Score")
ax4.set_ylabel("Count")
ax4.text(0.02, ax4.get_ylim()[1]*0.8, "Missed the Cone\n(Score ~0)", color='red', fontweight='bold')
ax4.text(0.75, ax4.get_ylim()[1]*0.8, "Hit the Cone\n(Score ~1)", color='red', fontweight='bold')
ax4.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as colors

# --- 1. DATA GENERATION ---
def generate_data(n_points=500, mode='cnn'):
    if mode == 'cnn':
        data = np.random.randn(n_points, 3)
    elif mode == 'transformer':
        # Anisotropic Cone
        mean_vec = np.array([1.0, 1.0, 1.0])
        noise = np.random.randn(n_points, 3) * 0.2
        data = mean_vec + noise
    
    norms = np.linalg.norm(data, axis=1, keepdims=True)
    return data / norms

def get_random_cavs(n_cavs=1000):
    cavs = np.random.randn(n_cavs, 3)
    norms = np.linalg.norm(cavs, axis=1, keepdims=True)
    return cavs / norms

# --- 2. CALCULATE SCORES ---
def calculate_tcav_scores(gradients, cavs):
    scores = []
    for cav in cavs:
        sensitivities = np.dot(gradients, cav)
        score = np.sum(sensitivities > 0) / len(sensitivities)
        scores.append(score)
    return np.array(scores)

# --- 3. RUN SIMULATION ---
n_grads = 500
n_cavs = 2000 
n_plot_arrows = 100 # Plot slightly more arrows to see the effect better

grads_cnn = generate_data(n_grads, mode='cnn')
grads_transformer = generate_data(n_grads, mode='transformer')
random_cavs = get_random_cavs(n_cavs)

scores_cnn = calculate_tcav_scores(grads_cnn, random_cavs)
scores_transformer = calculate_tcav_scores(grads_transformer, random_cavs)

# --- 4. PLOTTING SETUP ---
fig = plt.figure(figsize=(18, 12))
# Define Colormap: Blue (0.0) -> White (0.5) -> Red (1.0)
cmap = plt.cm.coolwarm
norm = colors.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# --- PLOT A: CNN (ISOTROPIC) ---
ax1 = fig.add_subplot(2, 2, 1, projection='3d')
ax1.scatter(grads_cnn[:,0], grads_cnn[:,1], grads_cnn[:,2], 
            c='black', alpha=0.1, s=5, label='Gradients (Data)')

# Plot Arrows colored by score
for i in range(n_plot_arrows):
    v = random_cavs[i]
    score = scores_cnn[i]
    # Color depends on score
    c = cmap(norm(score))
    ax1.quiver(0,0,0, v[0], v[1], v[2], color=c, length=1.2, arrow_length_ratio=0.1, alpha=0.8)

ax1.set_title("CNN: Isotropic 'Ball'\n(Arrows are mostly neutral)", fontsize=14, fontweight='bold')
ax1.set_xlim([-1,1]); ax1.set_ylim([-1,1]); ax1.set_zlim([-1,1])

# --- PLOT B: TRANSFORMER (ANISOTROPIC) ---
ax2 = fig.add_subplot(2, 2, 2, projection='3d')
ax2.scatter(grads_transformer[:,0], grads_transformer[:,1], grads_transformer[:,2], 
            c='purple', alpha=0.3, s=10, label='Gradients (Data)')

# Plot Arrows colored by score
for i in range(n_plot_arrows):
    v = random_cavs[i]
    score = scores_transformer[i]
    c = cmap(norm(score))
    # Make aligned/anti-aligned arrows slightly thicker to see them better
    lw = 2 if (score > 0.8 or score < 0.2) else 1 
    ax2.quiver(0,0,0, v[0], v[1], v[2], color=c, length=1.2, arrow_length_ratio=0.1, linewidth=lw, alpha=0.8)

ax2.set_title("Transformer: Anisotropic 'Cone'\n(Arrows split Red vs Blue)", fontsize=14, fontweight='bold')
ax2.set_xlim([-1,1]); ax2.set_ylim([-1,1]); ax2.set_zlim([-1,1])

# --- PLOT C: CNN SCORES ---
ax3 = fig.add_subplot(2, 2, 3)
sns.histplot(scores_cnn, bins=50, color='grey', kde=True, ax=ax3, stat='density')
ax3.axvline(0.5, color='black', linestyle='--')
ax3.set_title("CNN Score Distribution", fontsize=14)
ax3.set_xlabel("TCAV Score")

# --- PLOT D: TRANSFORMER SCORES ---
ax4 = fig.add_subplot(2, 2, 4)
sns.histplot(scores_transformer, bins=50, color='purple', kde=True, ax=ax4, stat='density')
ax4.axvline(0.5, color='black', linestyle='--')
ax4.set_title("Transformer Score Distribution", fontsize=14)
ax4.set_xlabel("TCAV Score")

# Add Colorbar to explain arrows
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7]) # Position on right
cb = fig.colorbar(sm, cax=cbar_ax)
cb.set_label("TCAV Score (Arrow Sensitivity)", fontsize=12)

plt.subplots_adjust(right=0.9)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D

# Settings
dim = 100000         # High dimensionality (like BERT/RoBERTa)
n_samples = 500    # Number of gradients/CAVs
seed = 42
np.random.seed(seed)

def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / (norms + 1e-9)

# 1. Generate "Fuzzy Ball" (Uniform Random Vectors)
def generate_fuzzy_ball(n, d):
    # Gaussian samples normalized -> Uniform on Hypersphere
    vecs = np.random.randn(n, d)
    return normalize(vecs)

# 2. Generate "Cone" (Directional vectors with low variance)
def generate_cone(n, d, noise_level=4.0):
    # Fixed mean direction
    mean_dir = np.random.randn(1, d)
    mean_dir = mean_dir / np.linalg.norm(mean_dir)
    
    # Add noise to create "cone" width
    noise = np.random.randn(n, d) * noise_level
    vecs = mean_dir + noise
    return normalize(vecs)

# 3. Calculate TCAV Scores
def get_tcav_scores(grads, cavs):
    # Score = fraction of grads with positive dot product with CAV
    # Dot product: (n_cavs, dim) @ (dim, n_grads) -> (n_cavs, n_grads)
    dots = cavs @ grads.T
    # Calculate fraction > 0 for each CAV
    scores = np.mean(dots > 0, axis=1)
    return scores

# --- Simulation 1: Fuzzy Ball vs Fuzzy Ball (Early Layers) ---
grads_fuzzy = generate_fuzzy_ball(n_samples, dim)
cavs_fuzzy = generate_fuzzy_ball(n_samples, dim)
scores_fuzzy = get_tcav_scores(grads_fuzzy, cavs_fuzzy)

# --- Simulation 2: Cone vs Fuzzy Ball (Deep Layers) ---
# Gradients are clustered (Cone), CAVs are random (Fuzzy Ball)
grads_cone = generate_cone(n_samples, dim, noise_level=0.004) 
cavs_random = generate_fuzzy_ball(n_samples, dim) 
scores_cone = get_tcav_scores(grads_cone, cavs_random)

# --- PCA for Visualization (3 Components) ---
# Fit PCA on combined data to share the coordinate space
all_vecs_fuzzy = np.vstack([grads_fuzzy, cavs_fuzzy])
pca_fuzzy = PCA(n_components=3).fit(all_vecs_fuzzy)
g_f_pca = pca_fuzzy.transform(grads_fuzzy)
c_f_pca = pca_fuzzy.transform(cavs_fuzzy)

all_vecs_cone = np.vstack([grads_cone, cavs_random])
pca_cone = PCA(n_components=3).fit(all_vecs_cone)
g_c_pca = pca_cone.transform(grads_cone)
c_c_pca = pca_cone.transform(cavs_random)

# --- Plotting ---
fig = plt.figure(figsize=(16, 10))

# Plot 1: Fuzzy 3D
ax1 = fig.add_subplot(2, 2, 1, projection='3d')
ax1.scatter(g_f_pca[:,0], g_f_pca[:,1], g_f_pca[:,2], c='blue', alpha=0.3, label='Gradients')
ax1.scatter(c_f_pca[:,0], c_f_pca[:,1], c_f_pca[:,2], c='red', alpha=0.3, label='Random CAVs')
ax1.set_title('Scenario 1: Fuzzy Gradients vs Random CAVs\n(Early Layers)')
ax1.legend()

# Plot 2: Fuzzy Scores
ax2 = fig.add_subplot(2, 2, 2)
ax2.hist(scores_fuzzy, bins=20, color='skyblue', edgecolor='black')
ax2.set_title('TCAV Scores (Fuzzy vs Fuzzy)\nExpected: Normal-ish around 0.5')
ax2.set_xlabel('Score')
ax2.set_xlim(0, 1)

# Plot 3: Cone 3D
ax3 = fig.add_subplot(2, 2, 3, projection='3d')
ax3.scatter(g_c_pca[:,0], g_c_pca[:,1], g_c_pca[:,2], c='green', alpha=0.6, label='Gradients (Cone)')
ax3.scatter(c_c_pca[:,0], c_c_pca[:,1], c_c_pca[:,2], c='red', alpha=0.1, label='Random CAVs')
ax3.set_title('Scenario 2: Cone Gradients vs Random CAVs\n(Deep Layers)')
ax3.legend()

# Plot 4: Cone Scores
ax4 = fig.add_subplot(2, 2, 4)
ax4.hist(scores_cone, bins=20, color='salmon', edgecolor='black')
ax4.set_title('TCAV Scores (Cone vs Fuzzy)\nExpected: Bi-modal (0 or 1)')
ax4.set_xlabel('Score')
ax4.set_xlim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

def get_bimodality_coefficient(dataset):
    """Computes Sarle's Bimodality Coefficient."""
    # BC = (skew^2 + 1) / kurtosis
    # Note: Scipy kurtosis is "excess" (Fisher), so we add 3 to get raw kurtosis
    s = skew(dataset)
    k = kurtosis(dataset, fisher=False) 
    return (s**2 + 1) / k

def simulate_phase_transition(dims, spreads, n_samples=500, n_runs=100):
    """
    dims: List of dimensions to test
    spreads: List of noise levels (angular variance proxies)
    """
    heatmap_data = np.zeros((len(spreads), len(dims)))
    
    for i, spread in enumerate(spreads):
        for j, d in enumerate(dims):
            # 1. Generate Gradients (Target Class)
            # Mean vector [1, 0, ... 0]
            mean_grad = np.zeros((1, d))
            mean_grad[0, 0] = 1
            
            # Add noise to create cone
            noise = np.random.randn(n_samples, d) * spread
            gradients = mean_grad + noise
            # Normalize to project onto hypersphere
            gradients /= np.linalg.norm(gradients, axis=1, keepdims=True)
            
            # 2. Generate Random CAVs
            cavs = np.random.randn(n_runs, d)
            cavs /= np.linalg.norm(cavs, axis=1, keepdims=True)
            
            # 3. Compute TCAV Scores
            # (n_runs, d) @ (d, n_samples) -> (n_runs, n_samples)
            dots = cavs @ gradients.T
            scores = np.mean(dots > 0, axis=1)
            
            # 4. Measure Bimodality
            # BC varies from 0 to 1. 1 = Perfect Bernoulli (two spikes). 
            # 5/9 (~0.555) is the uniform distribution. Normal is 1/3 (~0.33).
            bc = get_bimodality_coefficient(scores)
            heatmap_data[i, j] = bc
            
    return heatmap_data

# --- Experiment Setup ---
# Logarithmic scales for systematic view
dimensions = [10, 50, 200, 768, 2000, 10000] # From LaBraM pooled to Flattened CNN
spreads = [0.1, 0.3, 0.5, 0.8, 1.2, 2.0]     # From Tight Cone to Fuzzy Ball

# Run
results = simulate_phase_transition(dimensions, spreads)

# --- Plotting ---
plt.figure(figsize=(10, 8))
ax = sns.heatmap(results, annot=True, fmt=".2f", 
                 xticklabels=dimensions, yticklabels=spreads,
                 cmap="coolwarm", cbar_kws={'label': 'Bimodality Coeff (Low=Normal, High=Bi-modal)'})

plt.title("TCAV Distribution Phase Transition\n(Normal < 0.35 | Bi-modal > 0.55)")
plt.xlabel("Dimensionality (D)")
plt.ylabel("Gradient Spread (Noise Level)")
plt.gca().invert_yaxis() # Put low spread at bottom
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import ipywidgets as widgets
from IPython.display import display

# --- Simulation Core ---
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / (norms + 1e-10)

def generate_data(dim, noise_level, n_samples=300):
    # 1. Generate "Cone" Gradients (Signal)
    # Define a random mean direction
    mean_dir = np.random.randn(1, dim)
    mean_dir = mean_dir / np.linalg.norm(mean_dir)
    
    # Add noise to create the cone width
    # We scale noise relative to the dimension to keep "visual" consistency
    # (In high dims, distance metrics behave differently, so we scale slightly)
    scaled_noise = noise_level * (1.0) 
    grad_noise = np.random.randn(n_samples, dim) * scaled_noise
    
    grads = mean_dir + grad_noise
    grads = normalize(grads)

    # 2. Generate Random CAVs (Fuzzy Ball / Noise)
    cavs = np.random.randn(n_samples, dim)
    cavs = normalize(cavs)
    
    return grads, cavs

def calculate_scores(grads, cavs):
    # Dot product: (n_cavs, dim) @ (dim, n_grads) -> (n_cavs, n_grads)
    dots = cavs @ grads.T
    # Fraction of grads > 0 for each CAV
    scores = np.mean(dots > 0, axis=1)
    return scores

# --- Visualization ---
def update_plot(log_dim, log_noise):
    # Convert log-scale sliders to actual values
    dim = int(10**log_dim)
    noise_level = 10**log_noise
    
    # Run Simulation
    grads, cavs = generate_data(dim, noise_level)
    scores = calculate_scores(grads, cavs)
    
    # Setup Plots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    plt.suptitle(f"Dimensions: {dim}  |  Gradient Variance (Noise): {noise_level:.2f}", fontsize=16)
    
    # --- Plot 1: PCA Projection (Geometry) ---
    # Combine to find a shared subspace
    combined = np.vstack([grads, cavs])
    
    # Optimization: If dims are massive, PCA is slow. 
    # But since N=600 total, sklearn handles this efficiently even for D=10000
    pca = PCA(n_components=2)
    proj = pca.fit_transform(combined)
    
    g_proj = proj[:len(grads)]
    c_proj = proj[len(grads):]
    
    ax1.scatter(c_proj[:, 0], c_proj[:, 1], c='red', alpha=0.3, label='Random CAVs (Fuzzy Ball)')
    ax1.scatter(g_proj[:, 0], g_proj[:, 1], c='green', alpha=0.6, label='Gradients (Cone)')
    ax1.set_title("Geometry (First 2 PCs)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # --- Plot 2: TCAV Score Distribution ---
    ax2.hist(scores, bins=20, range=(0, 1), color='skyblue', edgecolor='black')
    ax2.set_title("TCAV Score Distribution")
    ax2.set_xlabel("TCAV Score")
    ax2.set_ylabel("Count")
    ax2.set_xlim(-0.05, 1.05)
    
    # Add annotations for interpretation
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    
    if std_score > 0.2:
        dist_type = "BIMODAL / UNIFORM"
        color = "red"
    elif std_score < 0.05 and (mean_score > 0.9 or mean_score < 0.1):
        dist_type = "COLLAPSED (Single Mode)"
        color = "darkred"
    else:
        dist_type = "NORMAL-ISH"
        color = "green"
        
    ax2.text(0.5, 0.9, dist_type, transform=ax2.transAxes, 
             ha='center', color=color, weight='bold', fontsize=12,
             bbox=dict(facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.show()

# --- Interactive Widgets ---
style = {'description_width': 'initial'}

slider_dim = widgets.FloatSlider(
    value=2.3,  # Starts around 200 dimensions
    min=1.0,    # 10^1 = 10 dims
    max=4.5,    # 10^4.5 ≈ 31,000 dims
    step=0.1,
    description='Log10 Dimensions:',
    style=style,
    continuous_update=False
)

slider_noise = widgets.FloatSlider(
    value=0.5,  # Starts moderate
    min=-0.01,   # 10^-1 = 0.1 (Very tight cone)
    max=2.0,    # 10^2 = 100 (Very fuzzy ball)
    step=0.001,
    description='Log10 Gradient Noise:',
    style=style,
    continuous_update=False
)

ui = widgets.HBox([slider_dim, slider_noise])
out = widgets.interactive_output(update_plot, {'log_dim': slider_dim, 'log_noise': slider_noise})

display(ui, out)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
import ipywidgets as widgets

def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / (norms + 1e-9)

def calculate_mean_angle(gradients, center_vector):
    # Clip dot products to handle potential floating point errors > 1.0
    dots = np.clip(gradients @ center_vector.T, -1.0, 1.0)
    angles_rad = np.arccos(dots)
    return np.degrees(np.mean(angles_rad))

def run_simulation(log_dim, log_noise):
    # Parameters from sliders
    dim = int(10**log_dim)
    noise_level = 10**log_noise
    n_samples = 300 # Keep sample size constant to isolate Dim/Noise effects
    
    np.random.seed(42)
    
    # 1. Generate The "Cone" (Gradients)
    # Start with a fixed center vector
    center_vec = np.random.randn(1, dim)
    center_vec = normalize(center_vec)
    
    # Add noise to create the cone
    # We add orthogonal noise to ensure the noise_level strictly controls width
    noise_vectors = np.random.randn(n_samples, dim)
    gradients = center_vec + (noise_vectors * noise_level)
    gradients = normalize(gradients)
    
    # 2. Generate Random CAVs (The "Fuzzy Ball")
    cavs = np.random.randn(n_samples, dim)
    cavs = normalize(cavs)
    
    # 3. Calculate TCAV Scores
    # Score = fraction of gradients that have positive dot product with CAV
    dots = cavs @ gradients.T
    scores = np.mean(dots > 0, axis=1)
    
    # 4. Calculate Cone Statistics
    avg_angle = calculate_mean_angle(gradients, center_vec)
    
    # --- Visualization ---
    fig = plt.figure(figsize=(18, 6))
    
    # Plot A: 3D PCA
    ax1 = fig.add_subplot(1, 3, 1, projection='3d')
    
    # PCA on combined data to share space
    combined = np.vstack([gradients, cavs])
    pca = PCA(n_components=3)
    combined_pca = pca.fit_transform(combined)
    g_pca = combined_pca[:n_samples]
    c_pca = combined_pca[n_samples:]
    
    ax1.scatter(g_pca[:,0], g_pca[:,1], g_pca[:,2], c='lime', s=20, alpha=0.6, label='Gradients (Cone)')
    ax1.scatter(c_pca[:,0], c_pca[:,1], c_pca[:,2], c='red', s=20, alpha=0.2, label='Random CAVs')
    
    # Plot the center vector arrow for reference (projected)
    center_pca = pca.transform(center_vec)
    ax1.quiver(0,0,0, center_pca[0,0], center_pca[0,1], center_pca[0,2], 
               color='black', length=np.max(g_pca), arrow_length_ratio=0.1, label='Cone Center')
    
    ax1.set_title(f"Geometry in 3D (PCA)\nTrue Dim: {dim}")
    ax1.legend(loc='upper right', fontsize='small')
    
    # Plot B: Histogram
    ax2 = fig.add_subplot(1, 3, 2)
    counts, bins, patches = ax2.hist(scores, bins=30, range=(0,1), color='skyblue', edgecolor='black')
    
    # Color code the histogram based on "Bi-modal vs Normal" feel
    ax2.set_xlim(-0.05, 1.05)
    ax2.set_xlabel("TCAV Score")
    ax2.set_ylabel("Count")
    ax2.set_title(f"TCAV Score Distribution\n(Noise: 10^{log_noise:.1f}, Dim: 10^{log_dim})")
    
    # Plot C: Angular Analysis text
    ax3 = fig.add_subplot(1, 3, 3)
    ax3.axis('off')
    
    text_str = (
        f"CONFIGURATION:\n"
        f"----------------\n"
        f"Dimensions: {dim}\n"
        f"Noise Level: {noise_level:.5f}\n\n"
        
        f"GEOMETRY STATS:\n"
        f"----------------\n"
        f"Cone Spread (Avg Angle): {avg_angle:.2f}°\n"
        f"Gradient Variance (Norm): {np.mean(np.var(gradients, axis=0)):.6f}\n\n"
        
        f"INTERPRETATION:\n"
        f"----------------\n"
    )
    
    # Logic for interpretation
    if avg_angle < 5:
        text_str += "State: COLLAPSED CONE\nGradients are nearly identical.\nExpect Bi-modal (0 or 1)."
    elif avg_angle > 80:
        text_str += "State: FUZZY BALL\nGradients are effectively random.\nExpect Normal (0.5)."
    else:
        # The transition zone
        if dim > 5000:
            text_str += "State: HIGH DIM TENSION\nCone is tight, but High Dim\nforces orthogonality.\nDistribution resists bi-modality."
        else:
            text_str += "State: LOW DIM ALIGNMENT\nDimensions are too low to\nhide the cone.\nDistribution splits to Bi-modal."

    ax3.text(0.1, 0.9, text_str, fontsize=12, verticalalignment='top', fontfamily='monospace')

    plt.tight_layout()
    plt.show()

# Create Controls
style = {'description_width': 'initial'}

interact(run_simulation, 
         log_dim=FloatSlider(value=2.3, min=1.0, max=4.7, step=0.1, 
                             description='Log10 Dimensions (10 - 50k)', style=style),
         log_noise=FloatSlider(value=0.0, min=-3.0, max=1.0, step=0.1, 
                               description='Log10 Noise (Tight -> Loose)', style=style));

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from ipywidgets import interact, FloatSlider, IntSlider
import ipywidgets as widgets

def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / (norms + 1e-9)

def generate_cone_at_angle(n, dim, noise_level, base_vector, angle_deg):
    # 1. Create a random vector orthogonal to base_vector
    random_vec = np.random.randn(1, dim)
    # Project out the base_vector component to make it orthogonal
    # Gram-Schmidt process
    proj = (random_vec @ base_vector.T) * base_vector
    orth_vec = random_vec - proj
    orth_vec = normalize(orth_vec)
    
    # 2. Rotate base_vector towards orth_vec by angle_deg
    # Formula: v_rot = v * cos(theta) + v_orth * sin(theta)
    theta = np.radians(angle_deg)
    cone_center = (base_vector * np.cos(theta)) + (orth_vec * np.sin(theta))
    cone_center = normalize(cone_center)
    
    # 3. Add noise to create the cone around this new center
    noise = np.random.randn(n, dim) * noise_level
    vectors = cone_center + noise
    return normalize(vectors)

def run_two_cone_simulation(log_dim, log_noise_grad, log_noise_cav, alignment_angle):
    # Parameters
    dim = int(10**log_dim)
    noise_g = 10**log_noise_grad
    noise_c = 10**log_noise_cav
    n_samples = 200 
    
    np.random.seed(42)
    
    # 1. Generate Gradient Cone (Fixed Center)
    grad_center = normalize(np.random.randn(1, dim))
    grad_noise_vecs = np.random.randn(n_samples, dim) * noise_g
    gradients = normalize(grad_center + grad_noise_vecs)
    
    # 2. Generate Concept CAV Cone (Relative to Gradient Center)
    cavs = generate_cone_at_angle(n_samples, dim, noise_c, grad_center, alignment_angle)
    
    # 3. Calculate TCAV Scores
    dots = cavs @ gradients.T
    scores = np.mean(dots > 0, axis=1)
    
    # 4. Geometry Stats
    # Calculate actual spread of the CAV cone
    cav_center_estimate = np.mean(cavs, axis=0, keepdims=True)
    cav_spread_dots = np.clip(cavs @ normalize(cav_center_estimate).T, -1.0, 1.0)
    cav_spread_deg = np.degrees(np.mean(np.arccos(cav_spread_dots)))

    # --- Visualization ---
    fig = plt.figure(figsize=(18, 6))
    
    # Plot A: 3D PCA
    ax1 = fig.add_subplot(1, 3, 1, projection='3d')
    
    combined = np.vstack([gradients, cavs])
    # Subsample for PCA speed if dims are huge
    pca_data = combined
    
    pca = PCA(n_components=3)
    combined_pca = pca.fit_transform(pca_data)
    
    g_pca = combined_pca[:n_samples]
    c_pca = combined_pca[n_samples:]
    
    ax1.scatter(g_pca[:,0], g_pca[:,1], g_pca[:,2], c='lime', s=20, alpha=0.5, label='Gradients')
    ax1.scatter(c_pca[:,0], c_pca[:,1], c_pca[:,2], c='orange', s=20, alpha=0.5, label='Meaningful CAVs')
    
    # Plot Center Vectors
    gc_pca = pca.transform(grad_center)
    ax1.quiver(0,0,0, gc_pca[0,0], gc_pca[0,1], gc_pca[0,2], color='darkgreen', length=np.max(g_pca), arrow_length_ratio=0.1)
    
    ax1.set_title(f"Geometry (PCA)\nDim: {dim}, Angle: {alignment_angle}°")
    ax1.legend()
    
    # Plot B: Histogram
    ax2 = fig.add_subplot(1, 3, 2)
    ax2.hist(scores, bins=30, range=(0,1), color='mediumpurple', edgecolor='black')
    ax2.set_xlim(-0.05, 1.05)
    ax2.set_xlabel("TCAV Score")
    ax2.set_title("TCAV Score Distribution")
    
    # Plot C: Interpretation
    ax3 = fig.add_subplot(1, 3, 3)
    ax3.axis('off')
    
    interp = "INTERPRETATION:\n----------------\n"
    if alignment_angle < 30:
        interp += "ALIGNED (Relevant)\nCones overlap.\nScores should be high (near 1.0)."
    elif alignment_angle > 150:
        interp += "OPPOSED (Negative)\nCones point away.\nScores should be low (near 0.0)."
    else:
        # The Orthogonal Zone
        interp += "ORTHOGONAL (Irrelevant?)\n"
        if dim < 500:
            interp += "Low Dim Artifacts:\nBi-modality likely.\nFalse confidence in scores."
        else:
            interp += "High Dim Stability:\nScores may drift to 0.5\nif spread is sufficient."
            
    stats = (
        f"\n\nSTATS:\n"
        f"Gradient Spread (Noise): 10^{log_noise_grad:.1f}\n"
        f"CAV Spread (Noise): 10^{log_noise_cav:.1f}\n"
        f"CAV Cone Width: ~{cav_spread_deg:.1f}°"
    )
            
    ax3.text(0.1, 0.9, interp + stats, fontsize=12, verticalalignment='top', fontfamily='monospace')
    
    plt.tight_layout()
    plt.show()

style = {'description_width': 'initial'}

interact(run_two_cone_simulation, 
         log_dim=FloatSlider(value=2.3, min=1.0, max=4.7, step=0.1, description='Log Dim (10-50k)'),
         log_noise_grad=FloatSlider(value=-0.5, min=-3.0, max=1.0, step=0.1, description='Grad Noise'),
         log_noise_cav=FloatSlider(value=-0.5, min=-3.0, max=1.0, step=0.1, description='CAV Noise'),
         alignment_angle=IntSlider(value=90, min=0, max=180, step=5, description='Angle (Deg)'));

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from ipywidgets import interact, FloatSlider, IntSlider
import ipywidgets as widgets

def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    # Avoid division by zero
    return vectors / (norms + 1e-9)

def generate_cone_at_angle(n, dim, noise_level, base_vector, angle_deg):
    """
    Generates a cone of vectors centered at a specific angle relative to a base_vector.
    """
    # 1. Create a random vector orthogonal to base_vector using Gram-Schmidt
    random_vec = np.random.randn(1, dim)
    proj = (random_vec @ base_vector.T) * base_vector
    orth_vec = random_vec - proj
    orth_vec = normalize(orth_vec)
    
    # 2. Rotate base_vector towards orth_vec by angle_deg
    theta = np.radians(angle_deg)
    cone_center = (base_vector * np.cos(theta)) + (orth_vec * np.sin(theta))
    cone_center = normalize(cone_center)
    
    # 3. Add noise to create the cone around this new center
    noise = np.random.randn(n, dim) * noise_level
    vectors = cone_center + noise
    return normalize(vectors)

def run_two_cone_simulation(log_dim, log_noise_grad, log_noise_cav, alignment_angle):
    # --- Setup Parameters ---
    dim = int(10**log_dim)
    noise_g = 10**log_noise_grad
    noise_c = 10**log_noise_cav
    n_samples = 200 
    
    np.random.seed(42)
    
    # --- Data Generation ---
    
    # 1. Gradient Cone (Fixed Center)
    grad_center = normalize(np.random.randn(1, dim))
    grad_noise_vecs = np.random.randn(n_samples, dim) * noise_g
    gradients = normalize(grad_center + grad_noise_vecs)
    
    # 2. CAV Cone (Relative to Gradient Center)
    cavs = generate_cone_at_angle(n_samples, dim, noise_c, grad_center, alignment_angle)
    
    # --- Calculations ---
    
    # TCAV Scores (Dot product positive?)
    dots = cavs @ gradients.T
    scores = np.mean(dots > 0, axis=1)
    
    # Stats Calculation
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    
    # Calculate geometric spread (Cone width)
    cav_center_estimate = np.mean(cavs, axis=0, keepdims=True)
    cav_spread_dots = np.clip(cavs @ normalize(cav_center_estimate).T, -1.0, 1.0)
    cav_spread_deg = np.degrees(np.mean(np.arccos(cav_spread_dots)))

    # --- Visualization ---
    fig = plt.figure(figsize=(20, 6))
    
    # Plot A: 3D PCA
    ax1 = fig.add_subplot(1, 3, 1, projection='3d')
    
    combined = np.vstack([gradients, cavs])
    pca = PCA(n_components=3)
    combined_pca = pca.fit_transform(combined)
    
    g_pca = combined_pca[:n_samples]
    c_pca = combined_pca[n_samples:]
    
    ax1.scatter(g_pca[:,0], g_pca[:,1], g_pca[:,2], c='lime', s=20, alpha=0.5, label='Gradients')
    ax1.scatter(c_pca[:,0], c_pca[:,1], c_pca[:,2], c='orange', s=20, alpha=0.5, label='CAVs')
    
    # Draw center vector for visual reference
    gc_pca = pca.transform(grad_center)
    ax1.quiver(0,0,0, gc_pca[0,0], gc_pca[0,1], gc_pca[0,2], color='darkgreen', length=np.max(g_pca), arrow_length_ratio=0.1, label='Grad Center')
    
    ax1.set_title(f"Geometry Visualization\n(PCA projection)", fontsize=10)
    ax1.legend()
    
    # Plot B: Histogram
    ax2 = fig.add_subplot(1, 3, 2)
    ax2.hist(scores, bins=20, range=(0,1), color='mediumpurple', edgecolor='black', alpha=0.7)
    ax2.axvline(mean_score, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_score:.2f}')
    ax2.set_xlim(-0.05, 1.05)
    ax2.set_xlabel("TCAV Score")
    ax2.set_title("Distribution of TCAV Scores", fontsize=10)
    ax2.legend()
    
    # Plot C: Dashboard / Stats
    ax3 = fig.add_subplot(1, 3, 3)
    ax3.axis('off')
    
    # 1. Configuration Section
    text_str = "SIMULATION PARAMETERS:\n"
    text_str += "----------------------\n"
    text_str += f"Dimensions (N) : {dim:,}\n"  # Comma formatting
    text_str += f"Angle Offset   : {alignment_angle}°\n"
    text_str += f"Grad Noise     : {noise_g:.4f}\n"
    text_str += f"CAV Noise      : {noise_c:.4f}\n\n"
    
    # 2. Results Section
    text_str += "RESULTS:\n"
    text_str += "--------\n"
    text_str += f"Mean Score     : {mean_score:.3f}\n"
    text_str += f"Score Std Dev  : {std_score:.3f}\n"
    text_str += f"Est. Cone Width: {cav_spread_deg:.1f}°\n\n"
    
    # 3. Dynamic Interpretation Section
    text_str += "INTERPRETATION:\n"
    text_str += "---------------\n"
    
    # Logic for interpretation
    if alignment_angle < 30:
        text_str += "state: POSITIVE ALIGNMENT\n"
        text_str += "> Cones overlap significantly.\n"
        text_str += "> Concept is interpreted as related.\n"
    elif alignment_angle > 150:
        text_str += "state: NEGATIVE ALIGNMENT\n"
        text_str += "> Cones point in opposite directions.\n"
        text_str += "> Concept is interpreted as opposed.\n"
    else:
        text_str += "state: ORTHOGONAL / NEUTRAL\n"
        
        # Check stability based on actual Std Dev
        if std_score < 0.1:
            text_str += "> STABLE RESULTS.\n"
            text_str += "> In High Dimensions, orthogonal vectors\n"
            text_str += "  reliably yield scores near 0.5.\n"
        elif std_score > 0.25:
            text_str += "> UNSTABLE / NOISY.\n"
            text_str += "> High variance detected.\n"
            if dim < 100:
                text_str += "> Likely due to LOW DIMENSIONS.\n"
                text_str += "  (Law of large numbers hasn't kicked in).\n"
            else:
                text_str += "> Likely due to EXTREME NOISE levels.\n"
        else:
            text_str += "> Moderate variance.\n"

    ax3.text(0.05, 0.95, text_str, fontsize=12, verticalalignment='top', fontfamily='monospace', linespacing=1.4)
    
    plt.tight_layout()
    plt.show()

# Run the interact
interact(run_two_cone_simulation, 
         log_dim=FloatSlider(value=2.3, min=1.0, max=5.0, step=0.1, description='Log Dim'),
         log_noise_grad=FloatSlider(value=-0.5, min=-3.0, max=1.0, step=0.1, description='Grad Noise'),
         log_noise_cav=FloatSlider(value=-0.5, min=-3.0, max=1.0, step=0.1, description='CAV Noise'),
         alignment_angle=IntSlider(value=90, min=0, max=180, step=5, description='Angle (Deg)'));

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

#set random seed for reproducibility
np.random.seed(42)

# --- Configuration ---
NUM_ACTIVATIONS = 1000
NUM_RANDOM_CAVS = 100  # Increased to 100
# Change this angle to simulate the "constant gradient" issue.
# Small angle (e.g., 15) = Tight cone = Bi-modal scores (0 or 1).
# Large angle (e.g., 90) = Spread out = Normal distribution around 0.5.
CONE_ANGLE_DEG = 70
TARGET_DIRECTION = np.array([1, 1, 1]) / np.sqrt(3)

# --- Helper Functions ---
def normalize(v):
    norm = np.linalg.norm(v)
    if norm == 0: return v
    return v / norm

def generate_cone_vectors(target_dir, angle_deg, num_samples):
    """Generates unit vectors within a cone around target_dir."""
    w = normalize(target_dir)
    if np.allclose(w, [0, 0, 1]):
        u = np.array([1, 0, 0])
    else:
        u = normalize(np.cross(w, [0, 0, 1]))
    v = np.cross(w, u)

    vectors = []
    angle_rad = np.radians(angle_deg)

    for _ in range(num_samples):
        theta = np.random.uniform(0, angle_rad)
        phi = np.random.uniform(0, 2 * np.pi)

        local_x = np.sin(theta) * np.cos(phi)
        local_y = np.sin(theta) * np.sin(phi)
        local_z = np.cos(theta)

        vec = local_x * u + local_y * v + local_z * w
        vectors.append(vec)

    return np.array(vectors)

def generate_random_vectors(num_samples):
    vecs = np.random.normal(0, 1, (num_samples, 3))
    return vecs / np.linalg.norm(vecs, axis=1)[:, np.newaxis]

# --- 1. Data Generation ---
activations = generate_cone_vectors(TARGET_DIRECTION, CONE_ANGLE_DEG, NUM_ACTIVATIONS)
random_cavs = generate_random_vectors(NUM_RANDOM_CAVS)

# --- 2. Visualization 1: 3D Space (CENTERED) ---
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot Activations (Cone)
ax.scatter(activations[:, 0], activations[:, 1], activations[:, 2],
           c='blue', alpha=0.1, s=5, label='Target Activations (Cone)')

# Plot Target Direction (Red Arrow)
ax.quiver(0, 0, 0, TARGET_DIRECTION[0], TARGET_DIRECTION[1], TARGET_DIRECTION[2],
          color='red', length=1.2, linewidth=3, label='Target Center')

# Plot Random CAVs (Gray Dashed Arrows) - Limit to first 20
num_cavs_to_plot = min(NUM_RANDOM_CAVS, 20)
for i in range(num_cavs_to_plot):
    cav = random_cavs[i]
    ax.quiver(0, 0, 0, cav[0], cav[1], cav[2],
              color='gray', length=1.0, alpha=0.3)

# Plot Centering
ax.set_xlim([-1, 1])
ax.set_ylim([-1, 1])
ax.set_zlim([-1, 1])
ax.set_box_aspect([1, 1, 1])

# Axes lines
ax.plot([-1, 1], [0, 0], [0, 0], 'k-', lw=0.5, alpha=0.3)
ax.plot([0, 0], [-1, 1], [0, 0], 'k-', lw=0.5, alpha=0.3)
ax.plot([0, 0], [0, 0], [-1, 1], 'k-', lw=0.5, alpha=0.3)

ax.set_title(f"3D View: Target Cone (Var ~ {CONE_ANGLE_DEG}°) & Random CAVs (First {num_cavs_to_plot} shown)")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.legend()
plt.show()

# --- 3. Compute Metrics ---
sensitivities = np.dot(activations, random_cavs.T)
tcav_scores = np.mean(sensitivities > 0, axis=0)

# --- 4. Visualization 2: Sensitivity Histograms (Subset) ---
# Only plot the first 10
num_plots = min(NUM_RANDOM_CAVS, 10)
fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
axes = axes.flatten()

for i in range(num_plots):
    data = sensitivities[:, i]
    median_val = np.median(data)

    axes[i].hist(data, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    axes[i].axvline(median_val, color='red', linestyle='dashed', linewidth=1.5, label=f'Med: {median_val:.2f}')
    axes[i].axvline(0, color='black', linewidth=1)
    axes[i].set_title(f"Rand CAV {i+1}")
    axes[i].legend(fontsize='small')

fig.suptitle(f"Sensitivity Distributions for First {num_plots} Random CAVs\n(Cone Variance: {CONE_ANGLE_DEG}°)", fontsize=16)
plt.tight_layout()
plt.show()

# --- 5. Visualization 3: Cosine Similarity Check (First CAV) ---
plt.figure(figsize=(6, 4))
plt.hist(sensitivities[:, 0], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
plt.title("Cosine Similarity Check (CAV 1)")
plt.xlabel("Cosine Similarity")
plt.ylabel("Frequency")
plt.show()

# --- 6. Visualization 4: TCAV Score Histogram (All 100) ---
plt.figure(figsize=(8, 5))
plt.hist(tcav_scores, bins=np.linspace(0, 1, 11), color='orange', edgecolor='black', alpha=0.7)
plt.axvline(0.5, color='black', linestyle='--')
plt.title(f"Distribution of TCAV Scores for {NUM_RANDOM_CAVS} Random CAVs\n(Target Cone Variance: {CONE_ANGLE_DEG}°)")
plt.xlabel("TCAV Score")
plt.ylabel("Count of CAVs")
plt.xlim(-0.05, 1.05)
plt.show()

print(f"Summary Statistics for {NUM_RANDOM_CAVS} TCAV Scores:")
print(f"Mean: {np.mean(tcav_scores):.2f}")
print(f"Std Dev: {np.std(tcav_scores):.2f}")
print(f"Min: {np.min(tcav_scores):.2f}, Max: {np.max(tcav_scores):.2f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- Configuration ---
NUM_ACTIVATIONS = 1000
NUM_RANDOM_CAVS = 1000
CONE_ANGLE_DEG = 90   # Fixed variance
NUM_DIMENSIONS = 100  # Increased dimensionality (Try 3 vs 100)

# Target direction (arbitrary unit vector in N-dim)
# We start with [1, 0, ... 0] for simplicity in generation
TARGET_DIRECTION = np.zeros(NUM_DIMENSIONS)
TARGET_DIRECTION[0] = 1

# --- Helper Functions ---
def normalize(v):
    norm = np.linalg.norm(v)
    if norm == 0: return v
    return v / norm

def generate_cone_high_dim(target_dir, angle_deg, num_samples, num_dim):
    """
    Generates vectors in N-dim space within a cone of angle_deg.
    Strategy: 
    1. Start with a vector aligned with target_dir.
    2. Perturb it by adding a random orthogonal vector scaled to achieve the angle.
    """
    vectors = []
    target_dir = normalize(target_dir)
    
    # Pre-calculate the scale factors for the desired angle
    # We want dot_product(v, target) = cos(angle)
    # v = cos(a)*target + sin(a)*random_ortho
    
    for _ in range(num_samples):
        # Pick a random angle uniformly within [0, angle_deg]
        # Note: In high dim, points naturally cluster at the edge of the cone,
        # but for this test, uniform angle selection is fine to represent "variance".
        angle_rad = np.radians(np.random.uniform(0, angle_deg))
        
        # Generate a random vector
        rand_v = np.random.normal(0, 1, num_dim)
        
        # Make it orthogonal to target_dir (Gram-Schmidt)
        proj = np.dot(rand_v, target_dir) * target_dir
        ortho = rand_v - proj
        ortho = normalize(ortho)
        
        # Combine to create vector at specific angle
        vec = np.cos(angle_rad) * target_dir + np.sin(angle_rad) * ortho
        vectors.append(vec)
        
    return np.array(vectors)

def generate_random_vectors(num_samples, num_dim):
    vecs = np.random.normal(0, 1, (num_samples, num_dim))
    return vecs / np.linalg.norm(vecs, axis=1)[:, np.newaxis]

# --- 1. Data Generation ---
activations = generate_cone_high_dim(TARGET_DIRECTION, CONE_ANGLE_DEG, NUM_ACTIVATIONS, NUM_DIMENSIONS)
random_cavs = generate_random_vectors(NUM_RANDOM_CAVS, NUM_DIMENSIONS)

# --- 2. Visualization 1: 3D Projection (First 3 Dimensions) ---
# We can only see a slice of the 100D space
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot Activations (Projected)
ax.scatter(activations[:, 0], activations[:, 1], activations[:, 2], 
           c='blue', alpha=0.3, s=5, label=f'Target Cone (First 3 of {NUM_DIMENSIONS} Dims)')

# Plot Target Direction
ax.quiver(0, 0, 0, TARGET_DIRECTION[0], TARGET_DIRECTION[1], TARGET_DIRECTION[2], 
          color='red', length=1.0, linewidth=3, label='Target Direction')

# Plot Random CAVs
# We plot just a few to show they point "randomly" in this projection
for i in range(min(20, NUM_RANDOM_CAVS)):
    cav = random_cavs[i]
    ax.quiver(0, 0, 0, cav[0], cav[1], cav[2], 
              color='gray', length=0.8, alpha=0.3, linestyle='--')

ax.set_xlim([-1, 1])
ax.set_ylim([-1, 1])
ax.set_zlim([-1, 1])
ax.set_title(f"Projection of {NUM_DIMENSIONS}-Dimensional Space\nCone Variance: {CONE_ANGLE_DEG}°")
ax.legend()
plt.show()

# --- 3. Compute Metrics ---
sensitivities = np.dot(activations, random_cavs.T)
tcav_scores = np.mean(sensitivities > 0, axis=0)

# --- 4. Visualization 2: Sensitivity Histograms (Subset) ---
num_plots = min(NUM_RANDOM_CAVS, 5)
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharex=True, sharey=True)
if num_plots == 1: axes = [axes]

for i in range(num_plots):
    data = sensitivities[:, i]
    mean_val = np.mean(data)
    
    axes[i].hist(data, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    axes[i].axvline(0, color='black', linewidth=1)
    axes[i].set_title(f"Rand CAV {i+1}\nMean Sens: {mean_val:.4f}")

fig.suptitle(f"Sensitivity Distributions (High Dim: {NUM_DIMENSIONS}D) - Note how centered they are on 0", fontsize=14)
plt.tight_layout()
plt.show()

# --- 5. Visualization 3: TCAV Score Histogram (All 100) ---
plt.figure(figsize=(8, 5))
plt.hist(tcav_scores, bins=np.linspace(0, 1, 11), color='purple', edgecolor='black', alpha=0.7)
plt.axvline(0.5, color='black', linestyle='--')
plt.title(f"Distribution of TCAV Scores ({NUM_DIMENSIONS} Dimensions)\nExpected: Cluster at 0.5 (Uni-modal)")
plt.xlabel("TCAV Score")
plt.ylabel("Count of CAVs")
plt.xlim(-0.05, 1.05)
plt.show()

print(f"Summary Statistics for {NUM_RANDOM_CAVS} TCAV Scores in {NUM_DIMENSIONS}D:")
print(f"Mean: {np.mean(tcav_scores):.2f}")
print(f"Std Dev: {np.std(tcav_scores):.2f}")
print(f"Min: {np.min(tcav_scores):.2f}, Max: {np.max(tcav_scores):.2f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
NUM_SAMPLES = 500
NUM_TEST_RUNS = 100 # Number of CAVs to test
DIMENSIONS = 100
CONE_ANGLE = 60     # Wide cone, as you noted 60 deg is still bimodal

# 1. Define Target Cone Center
target_dir = np.random.normal(0, 1, DIMENSIONS)
target_dir /= np.linalg.norm(target_dir)

# --- Helper: Generate Data ---
def generate_cone_data(center, angle_deg, n_points):
    """Generates data points clustered around 'center' vector within 'angle_deg'."""
    data = []
    # Simple rejection sampling for demonstration (inefficient but accurate for concept)
    # or projection method for speed
    
    # Using the projection method from previous turn for speed/stability
    for _ in range(n_points):
        rand_v = np.random.normal(0, 1, len(center))
        # Orthogonalize
        proj = np.dot(rand_v, center) * center
        ortho = rand_v - proj
        ortho /= np.linalg.norm(ortho)
        
        # Mix
        theta = np.radians(np.random.uniform(0, angle_deg))
        vec = np.cos(theta) * center + np.sin(theta) * ortho
        data.append(vec)
    return np.array(data)

def get_tcav_scores(activations, cav_pool):
    """Calculates TCAV scores for a pool of CAVs against activations."""
    # Sensitivity = Dot product
    sensitivities = np.dot(activations, cav_pool.T)
    # Score = Fraction > 0
    return np.mean(sensitivities > 0, axis=0)

# --- Simulation ---
# 1. Generate Target Class Activations (The "Cone")
activations = generate_cone_data(target_dir, CONE_ANGLE, NUM_SAMPLES)

# 2. Generate RANDOM CAVs (The Null Hypothesis)
random_cavs = np.random.normal(0, 1, (NUM_TEST_RUNS, DIMENSIONS))
random_cavs /= np.linalg.norm(random_cavs, axis=1, keepdims=True)

# 3. Generate "REAL CONCEPT" CAVs (The Alternative Hypothesis)
# Simulate a concept that is RELEVANT (aligned with target) but has some noise
# We create vectors that are within ~30 degrees of the target direction
concept_cavs = generate_cone_data(target_dir, 30, NUM_TEST_RUNS) 

# 4. Calculate Scores
random_scores = get_tcav_scores(activations, random_cavs)
concept_scores = get_tcav_scores(activations, concept_cavs)

# --- Visualization ---
plt.figure(figsize=(10, 6))

# Plot Random Distribution
plt.hist(random_scores, bins=20, range=(0, 1), color='red', alpha=0.5, 
         label=f'Random CAVs\n(Mean: {np.mean(random_scores):.2f}, Std: {np.std(random_scores):.2f})', edgecolor='black')

# Plot Concept Distribution
plt.hist(concept_scores, bins=20, range=(0, 1), color='green', alpha=0.5, 
         label=f'True Concept CAVs\n(Mean: {np.mean(concept_scores):.2f}, Std: {np.std(concept_scores):.2f})', edgecolor='black')

plt.axvline(0.5, color='black', linestyle='--', label='Random Chance (0.5)')
plt.title(f"Why TCAV Works Despite Bimodality\n(Target Cone Angle: {CONE_ANGLE}°, Dims: {DIMENSIONS})")
plt.xlabel("TCAV Score")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
# Using the "Problematic" settings from before
DIMENSIONS = 100
CONE_ANGLE = 15      # Tight cone (Linear Layer simulation)
NUM_SAMPLES = 500    # Activations
NUM_CAVS = 200       # Number of random CAVs to test

# Target direction (e.g., the 'Linear' gradient)
target_dir = np.zeros(DIMENSIONS)
target_dir[0] = 1

# --- Helper Functions ---
def normalize(v):
    norm = np.linalg.norm(v, axis=1, keepdims=True)
    return v / (norm + 1e-9)

def generate_cone(center, angle_deg, count):
    """Generates vectors in a cone around center."""
    # 1. Random orthogonal vectors
    rand = np.random.randn(count, len(center))
    proj = (rand @ center)[:, np.newaxis] * center
    ortho = normalize(rand - proj)
    
    # 2. Mix to create cone
    theta = np.radians(np.random.uniform(0, angle_deg, count))
    # Note: In high dim, 'uniform' angle isn't physically uniform density 
    # but works perfectly to simulate "variance" for this test.
    
    cone = np.outer(np.cos(theta), center) + (ortho * np.sin(theta)[:, np.newaxis])
    return cone

# --- Simulation ---
# 1. Generate Data (The "Constant" Gradients)
activations = generate_cone(target_dir, CONE_ANGLE, NUM_SAMPLES)

# 2. Generate Random CAVs
random_cavs = np.random.randn(NUM_CAVS, DIMENSIONS)
random_cavs = normalize(random_cavs)

# 3. Calculate BOTH Metrics
# Dot product matrix: (n_activations x n_cavs)
dot_products = activations @ random_cavs.T

# Metric A: Original TCAV Score (Binary Fraction)
tcav_scores = np.mean(dot_products > 0, axis=0)

# Metric B: Raw Sensitivity (Average Magnitude)
raw_scores = np.mean(dot_products, axis=0)

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot A: Original TCAV Score
axes[0].hist(tcav_scores, bins=20, range=(0, 1), color='red', alpha=0.7, edgecolor='black')
axes[0].set_title(f"Original TCAV Score\n(Bi-modal: The Problem)")
axes[0].set_xlabel("Score (Fraction > 0)")
axes[0].set_ylabel("Count of Random CAVs")
axes[0].axvline(0.5, color='black', linestyle='--')

# Plot B: Raw Sensitivity
axes[1].hist(raw_scores, bins=20, color='green', alpha=0.7, edgecolor='black')
axes[1].set_title(f"Alternative: Raw Sensitivity\n(Normal Distribution: The Fix)")
axes[1].set_xlabel("Score (Average Dot Product)")
axes[1].axvline(0, color='black', linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
DIMENSIONS = 100
TARGET_CONE_ANGLE = 15  # The "Tight Cone" (Linear Layer) scenario
NUM_SAMPLES = 1000      # Number of activations
NUM_RANDOM_CAVS = 500   # For the gray background
NUM_CONCEPT_CAVS = 200  # For the colored signal
CONCEPT_NOISE_DEG = 10  # Noise in the concept vector itself

# Target direction (gradient)
target_dir = np.zeros(DIMENSIONS)
target_dir[0] = 1

# --- Helper Functions ---
def normalize(v):
    norm = np.linalg.norm(v, axis=1, keepdims=True)
    return v / (norm + 1e-9)

def generate_cone(center, angle_deg, count):
    """Generates vectors in a cone around center with uniform-ish angular spread."""
    # 1. Random orthogonal vectors
    rand = np.random.randn(count, len(center))
    proj = (rand @ center)[:, np.newaxis] * center
    ortho = normalize(rand - proj)
    
    # 2. Mix to create cone. 
    # We use a normal distribution for angle to simulate "noisy measurement"
    # Centered at angle_deg, with some spread.
    theta_mean = np.radians(angle_deg)
    theta_std = np.radians(CONCEPT_NOISE_DEG) # 10 degree jitter
    theta = np.random.normal(theta_mean, theta_std, count)
    
    vecs = np.outer(np.cos(theta), center) + (ortho * np.sin(theta)[:, np.newaxis])
    return vecs

def get_raw_scores(activations, cavs):
    # Raw Sensitivity = Average Dot Product
    # activations: (N_samples, Dims)
    # cavs: (N_cavs, Dims)
    # Result: (N_cavs,)
    return np.mean(activations @ cavs.T, axis=0)

# --- 1. Setup Data ---
# Generate the fixed "Target Cone" (simulating the gradient)
# We use a uniform spread for the target cone itself
activations = generate_cone(target_dir, TARGET_CONE_ANGLE, NUM_SAMPLES)

# Generate the Baseline Random Distribution (Gray)
# Random vectors in high dim are approx 90 degrees from target
random_cavs = np.random.randn(NUM_RANDOM_CAVS, DIMENSIONS)
random_cavs = normalize(random_cavs)
random_dist = get_raw_scores(activations, random_cavs)
random_mean = np.mean(random_dist)
random_std = np.std(random_dist)

# --- 2. Run Sweep ---
# We want 9 steps from 180 (Opposite) to 0 (Aligned)
angles = np.linspace(180, 0, 9)

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, angle in enumerate(angles):
    ax = axes[i]
    
    # Generate Concept CAVs at this specific angle
    concept_cavs = generate_cone(target_dir, angle, NUM_CONCEPT_CAVS)
    concept_dist = get_raw_scores(activations, concept_cavs)
    
    # Calculate Statistics
    concept_mean = np.mean(concept_dist)
    z_score = (concept_mean - random_mean) / random_std
    
    # Plot Gray Background (Random Baseline)
    ax.hist(random_dist, bins=30, color='gray', alpha=0.3, density=True, label='Random (Baseline)')
    
    # Plot Colored Signal (Concept)
    # Color mapping: Red (Opposite) -> Yellow (Orthogonal) -> Green (Aligned)
    if angle > 100: color = 'tab:red'
    elif angle < 80: color = 'tab:green'
    else: color = 'tab:orange'
    
    ax.hist(concept_dist, bins=30, color=color, alpha=0.7, density=True, label=f'Concept (~{int(angle)}°)')
    
    # Decorations
    ax.axvline(random_mean, color='gray', linestyle='--', linewidth=1)
    ax.set_title(f"Angle: {int(angle)}° | Z-Score: {z_score:.1f}")
    if i == 0: ax.legend(loc='upper left', fontsize='small')
    
    # Set consistent x-axis to see the shift
    # Theoretical max dot product is 1.0, min is -1.0. 
    # But average over cone < 1.0. Let's fix to reasonable visual range.
    ax.set_xlim(-1.2, 1.2) 
    
    # Remove y-ticks for cleaner look
    ax.set_yticks([])

plt.suptitle(f"Raw Sensitivity Distributions: Transition from Opposite to Aligned\n(Target Cone: {TARGET_CONE_ANGLE}°, Dims: {DIMENSIONS})", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
DIMENSIONS = 100
TARGET_CONE_ANGLE = 15  # Tight cone (Linear Layer scenario)
NUM_SAMPLES = 1000      # Activations
NUM_RANDOM_CAVS = 500   # Gray background
NUM_CONCEPT_CAVS = 200  # Colored signal
CONCEPT_NOISE_DEG = 15  # Variance in the concept itself

# Target direction (gradient)
target_dir = np.zeros(DIMENSIONS)
target_dir[0] = 1

# --- Helper Functions ---
def normalize(v):
    norm = np.linalg.norm(v, axis=1, keepdims=True)
    return v / (norm + 1e-9)

def generate_cone(center, angle_deg, count):
    """Generates vectors in a cone around center."""
    # 1. Random orthogonal vectors
    rand = np.random.randn(count, len(center))
    proj = (rand @ center)[:, np.newaxis] * center
    ortho = normalize(rand - proj)
    
    # 2. Mix to create cone
    theta_mean = np.radians(angle_deg)
    theta_std = np.radians(CONCEPT_NOISE_DEG)
    theta = np.random.normal(theta_mean, theta_std, count)
    
    vecs = np.outer(np.cos(theta), center) + (ortho * np.sin(theta)[:, np.newaxis])
    return vecs

def get_raw_scores(activations, cavs):
    # Raw Sensitivity = Average Dot Product
    return np.mean(activations @ cavs.T, axis=0)

def get_tcav_scores(activations, cavs):
    # TCAV Score = Fraction of positive dot products
    # Shape: (N_samples, N_cavs)
    dot_products = activations @ cavs.T
    return np.mean(dot_products > 0, axis=0)

# --- 1. Setup Data ---
# Generate Target Class Activations (The "Gradient Cone")
activations = generate_cone(target_dir, TARGET_CONE_ANGLE, NUM_SAMPLES)

# Generate Baseline Random Distributions (Gray)
random_cavs = np.random.randn(NUM_RANDOM_CAVS, DIMENSIONS)
random_cavs = normalize(random_cavs)

# Pre-calculate Random Baselines
rand_raw_dist = get_raw_scores(activations, random_cavs)
rand_tcav_dist = get_tcav_scores(activations, random_cavs)

# --- 2. Run Sweep ---
# 9 steps from 180 (Opposite) to 0 (Aligned)
angles = np.linspace(180, 0, 9)

# Create a tall figure: Top half for Raw, Bottom half for TCAV
fig, axes = plt.subplots(6, 3, figsize=(15, 20))
plt.subplots_adjust(hspace=0.4)

for i, angle in enumerate(angles):
    # Generate Concept CAVs
    concept_cavs = generate_cone(target_dir, angle, NUM_CONCEPT_CAVS)
    
    # Calculate metrics
    concept_raw = get_raw_scores(activations, concept_cavs)
    concept_tcav = get_tcav_scores(activations, concept_cavs)
    
    # Determine color
    if angle > 100: color = 'tab:red'      # Opposite
    elif angle < 80: color = 'tab:green'   # Aligned
    else: color = 'tab:orange'             # Random/Orthogonal
    
    # --- PLOT 1: RAW SENSITIVITY (Rows 0-2) ---
    row_raw = i // 3
    col_raw = i % 3
    ax_raw = axes[row_raw, col_raw]
    
    ax_raw.hist(rand_raw_dist, bins=30, color='gray', alpha=0.3, density=True, label='Random')
    ax_raw.hist(concept_raw, bins=30, color=color, alpha=0.7, density=True, label=f'Concept ~{int(angle)}°')
    ax_raw.axvline(np.mean(rand_raw_dist), color='gray', linestyle='--')
    ax_raw.set_title(f"Raw Sens. (Angle {int(angle)}°)")
    ax_raw.set_xlim(-0.8, 0.8)
    ax_raw.set_yticks([])
    if i == 0: ax_raw.legend(loc='upper left', fontsize='small')

    # --- PLOT 2: TCAV SCORE (Rows 3-5) ---
    row_tcav = (i // 3) + 3
    col_tcav = i % 3
    ax_tcav = axes[row_tcav, col_tcav]
    
    ax_tcav.hist(rand_tcav_dist, bins=20, range=(0,1), color='gray', alpha=0.3, density=True, label='Random')
    ax_tcav.hist(concept_tcav, bins=20, range=(0,1), color=color, alpha=0.7, density=True, label=f'Concept ~{int(angle)}°')
    ax_tcav.axvline(0.5, color='black', linestyle='--') # Theoretical random center
    ax_tcav.set_title(f"TCAV Score (Angle {int(angle)}°)")
    ax_tcav.set_xlim(-0.05, 1.05)
    ax_tcav.set_yticks([])

# Add Section Headers
fig.text(0.5, 0.90, "Metric 1: Raw Sensitivity (Continuous, Normal Distributions)", 
         ha='center', fontsize=16, weight='bold')
fig.text(0.5, 0.48, "Metric 2: TCAV Score (Binary Fraction, Clustered/Bi-modal)", 
         ha='center', fontsize=16, weight='bold')

plt.tight_layout(rect=[0, 0.03, 1, 0.96]) # Make room for titles
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
DIMENSIONS = 100
TARGET_CONE_ANGLE = 15  # "Low Variance" / Linear Scenario
NUM_SAMPLES = 1000      # Number of activations
NUM_RANDOM_CAVS = 500   # Gray background
NUM_CONCEPT_CAVS = 200  # Colored signal
CONCEPT_NOISE_DEG = 10  # Measurement noise in the concept vector

# Target direction (gradient)
target_dir = np.zeros(DIMENSIONS)
target_dir[0] = 1

# --- Helper Functions ---
def normalize(v):
    norm = np.linalg.norm(v, axis=1, keepdims=True)
    return v / (norm + 1e-9)

def generate_cone(center, angle_deg, count):
    """Generates vectors in a cone around center."""
    # 1. Random orthogonal vectors
    rand = np.random.randn(count, len(center))
    proj = (rand @ center)[:, np.newaxis] * center
    ortho = normalize(rand - proj)
    
    # 2. Mix to create cone
    theta_mean = np.radians(angle_deg)
    theta_std = np.radians(CONCEPT_NOISE_DEG)
    theta = np.random.normal(theta_mean, theta_std, count)
    
    vecs = np.outer(np.cos(theta), center) + (ortho * np.sin(theta)[:, np.newaxis])
    return vecs

def get_raw_scores(activations, cavs):
    # Raw Sensitivity = Average Dot Product
    return np.mean(activations @ cavs.T, axis=0)

def get_tcav_scores(activations, cavs):
    # TCAV Score = Fraction of positive dot products
    dot_products = activations @ cavs.T
    return np.mean(dot_products > 0, axis=0)

# --- 1. Setup Data ---
# Target Class Activations (Tight Cone)
activations = generate_cone(target_dir, TARGET_CONE_ANGLE, NUM_SAMPLES)

# Baseline Random Distributions (Gray)
random_cavs = np.random.randn(NUM_RANDOM_CAVS, DIMENSIONS)
random_cavs = normalize(random_cavs)
rand_raw_dist = get_raw_scores(activations, random_cavs)
rand_tcav_dist = get_tcav_scores(activations, random_cavs)

# --- 2. Run Sweep ---
# 9 steps from 180 (Opposite) to 0 (Aligned)
angles = np.linspace(180, 0, 9)

# Create a tall figure: Top half for Raw, Bottom half for TCAV
fig, axes = plt.subplots(6, 3, figsize=(15, 20))
plt.subplots_adjust(hspace=0.4, wspace=0.3)

for i, angle in enumerate(angles):
    # Generate Concept CAVs
    concept_cavs = generate_cone(target_dir, angle, NUM_CONCEPT_CAVS)
    
    # Calculate metrics
    concept_raw = get_raw_scores(activations, concept_cavs)
    concept_tcav = get_tcav_scores(activations, concept_cavs)
    
    # Determine color
    if angle > 100: color = 'tab:red'      # Opposite
    elif angle < 80: color = 'tab:green'   # Aligned
    else: color = 'tab:orange'             # Random/Orthogonal
    
    # --- PLOT 1: RAW SENSITIVITY (Rows 0-2) ---
    row_raw = i // 3
    col_raw = i % 3
    ax_raw = axes[row_raw, col_raw]
    
    ax_raw.hist(rand_raw_dist, bins=30, color='gray', alpha=0.3, density=True, label='Random')
    ax_raw.hist(concept_raw, bins=30, color=color, alpha=0.7, density=True, label=f'Concept ~{int(angle)}°')
    ax_raw.axvline(np.mean(rand_raw_dist), color='gray', linestyle='--')
    
    # Calculate Z-Score
    z_score = (np.mean(concept_raw) - np.mean(rand_raw_dist)) / np.std(rand_raw_dist)
    ax_raw.set_title(f"Angle {int(angle)}° | Z: {z_score:.1f}")
    ax_raw.set_xlim(-0.8, 0.8)
    ax_raw.set_yticks([])
    if i == 0: ax_raw.legend(loc='upper left', fontsize='small')

    # --- PLOT 2: TCAV SCORE (Rows 3-5) ---
    row_tcav = (i // 3) + 3
    col_tcav = i % 3
    ax_tcav = axes[row_tcav, col_tcav]
    
    ax_tcav.hist(rand_tcav_dist, bins=20, range=(0,1), color='gray', alpha=0.3, density=True, label='Random')
    ax_tcav.hist(concept_tcav, bins=20, range=(0,1), color=color, alpha=0.7, density=True, label=f'Concept ~{int(angle)}°')
    ax_tcav.axvline(0.5, color='black', linestyle='--')
    ax_tcav.set_title(f"TCAV Score (Mean: {np.mean(concept_tcav):.2f})")
    ax_tcav.set_xlim(-0.05, 1.05)
    ax_tcav.set_yticks([])

# Add Section Headers
fig.text(0.5, 0.90, "Metric 1: Raw Sensitivity (Continuous, Normal Distributions)", 
         ha='center', fontsize=16, weight='bold')
fig.text(0.5, 0.48, "Metric 2: TCAV Score (Binary Fraction, Clustered/Bi-modal)", 
         ha='center', fontsize=16, weight='bold')

plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()